# PhoBERT Moderation — Evaluation & Ablation Study

Run this notebook on Colab/Kaggle after training to:
1. Evaluate PhoBERT on held-out test set
2. Compare against rules-only baseline
3. Ablation study on rule/ML weight and threshold
4. Error analysis (false positives / false negatives)

All metrics are logged to W&B project `fshop-moderation`.

**test_split.csv:** Nếu file bị mất (Colab session kết thúc), Cell 3 sẽ tự tạo lại từ unified_dataset.csv với cùng random_state=42.

In [ ]:
!pip install -q transformers datasets accelerate wandb scikit-learn huggingface_hub

In [ ]:
# ── Cell 2: Config — EDIT THESE ───────────────────────────────────────────
import os

HF_MODEL_REPO = "luonvuituoi71/fshop-phobert-moderation"  # CHANGE nếu khác
WANDB_API_KEY = "your-wandb-api-key"                       # CHANGE
UNIFIED_CSV   = "unified_dataset.csv"   # upload file này lên Colab
TEST_CSV      = "test_split.csv"        # tự tạo lại nếu không có

LABEL_NAMES = ["toxic", "hate_speech"]  # 2 labels — khớp với model đã train
MAX_LENGTH  = 256
BATCH_SIZE  = 32

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
print("Config loaded. Labels:", LABEL_NAMES)

In [ ]:
# ── Cell 3: Load hoặc tạo lại test_split ──────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

if Path(TEST_CSV).exists():
    print(f"Found {TEST_CSV} — loading directly.")
    test_df = pd.read_csv(TEST_CSV)
else:
    print(f"{TEST_CSV} not found — recreating from {UNIFIED_CSV} (random_state=42)...")
    df = pd.read_csv(UNIFIED_CSV)
    df = df[df["text"].notna() & (df["text"].str.strip() != "")].reset_index(drop=True)
    # Giữ đúng split như lúc train (stratify on 'toxic', same seed)
    _, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["toxic"])
    _, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["toxic"])
    test_df = test_df.reset_index(drop=True)
    test_df.to_csv(TEST_CSV, index=False)
    print(f"Saved {TEST_CSV} ({len(test_df)} rows)")

# Chỉ giữ labels model biết
available = [l for l in LABEL_NAMES if l in test_df.columns]
texts      = test_df["text"].tolist()
true_labels = test_df[available].values.astype(int)

print(f"\nTest samples: {len(test_df)}")
print(test_df[available].sum().to_string())

In [ ]:
# ── Cell 4: PhoBERT predictions ───────────────────────────────────────────
import torch
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import hf_hub_download

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_REPO)
model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_REPO).to(device)
model.eval()

try:
    threshold_path = hf_hub_download(HF_MODEL_REPO, "thresholds.json")
    with open(threshold_path) as f:
        thresholds = json.load(f)
    print("Thresholds loaded:", thresholds)
except Exception:
    thresholds = {lbl: 0.5 for lbl in LABEL_NAMES}
    print("thresholds.json not found — using default 0.5")

def predict_batch(texts_batch):
    enc = tokenizer(
        texts_batch, truncation=True, max_length=MAX_LENGTH,
        padding=True, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.sigmoid(logits).cpu().numpy()

all_probs = []
for i in range(0, len(texts), BATCH_SIZE):
    all_probs.append(predict_batch(texts[i:i+BATCH_SIZE]))
    if i % (BATCH_SIZE * 10) == 0:
        print(f"  {i}/{len(texts)}", end="\r")

ml_probs = np.vstack(all_probs)
print(f"\nPhoBERT predictions done. Shape: {ml_probs.shape}")

In [ ]:
# ── Cell 5: Evaluation metrics ────────────────────────────────────────────
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import wandb

wandb.init(project="fshop-moderation", name="eval-phobert-v1")

# Apply per-label thresholds từ thresholds.json
ml_preds = np.array([
    [(1 if ml_probs[i, j] >= thresholds.get(lbl, 0.5) else 0)
     for j, lbl in enumerate(LABEL_NAMES)]
    for i in range(len(texts))
])

macro_f1 = f1_score(true_labels, ml_preds, average="macro", zero_division=0)
print(f"\nMacro F1: {macro_f1:.4f}")
print(classification_report(true_labels, ml_preds, target_names=LABEL_NAMES, zero_division=0))

per_f1 = f1_score(true_labels, ml_preds, average=None, zero_division=0)
wandb.log({"macro_f1": macro_f1, **{f"f1_{l}": s for l, s in zip(LABEL_NAMES, per_f1)}})

try:
    auc = roc_auc_score(true_labels, ml_probs, average="macro")
    print(f"ROC-AUC (macro): {auc:.4f}")
    wandb.log({"roc_auc": auc})
except Exception as e:
    print(f"AUC skipped: {e}")

In [ ]:
# ── Cell 6: Rule-only baseline (toxic label) ──────────────────────────────
import re

TOXIC_PATTERN = re.compile(
    r"(đ[eéèẹê][oó]|địt|đụ|đmm?|vcl|vkl|lồn|ngu|óc\s*ch[oó]|f[u*@#!]+ck|sh[i!1]+t|b[i!1]+tch)",
    re.IGNORECASE
)

TOXIC_IDX = LABEL_NAMES.index("toxic")

rule_toxic_preds = np.array([1 if TOXIC_PATTERN.search(t) else 0 for t in texts])
rule_f1 = f1_score(true_labels[:, TOXIC_IDX], rule_toxic_preds, zero_division=0)
ml_f1   = f1_score(true_labels[:, TOXIC_IDX],
                   (ml_probs[:, TOXIC_IDX] >= thresholds.get("toxic", 0.5)).astype(int),
                   zero_division=0)

print(f"Rule-only  F1 (toxic): {rule_f1:.4f}")
print(f"PhoBERT    F1 (toxic): {ml_f1:.4f}")
print(f"Gain: +{(ml_f1 - rule_f1):.4f}")
wandb.log({"rule_only_f1_toxic": rule_f1, "phobert_f1_toxic": ml_f1})

In [ ]:
# ── Cell 7: Ablation — rule weight vs combined score ──────────────────────
from sklearn.metrics import recall_score

rule_scores = np.array([1.0 if TOXIC_PATTERN.search(t) else 0.0 for t in texts])

results = []
for rule_w in np.arange(0.0, 0.55, 0.1):
    ml_w = 1.0 - rule_w
    for thresh in np.arange(0.30, 0.75, 0.05):
        combined = rule_w * rule_scores + ml_w * ml_probs[:, TOXIC_IDX]
        preds    = (combined >= thresh).astype(int)
        f1   = f1_score(true_labels[:, TOXIC_IDX], preds, zero_division=0)
        fnr  = 1 - recall_score(true_labels[:, TOXIC_IDX], preds, zero_division=0)
        fpr  = (preds[true_labels[:, TOXIC_IDX] == 0].sum() /
                max((true_labels[:, TOXIC_IDX] == 0).sum(), 1))
        results.append({"rule_w": round(rule_w, 2), "thresh": round(float(thresh), 2),
                        "f1": round(f1, 4), "fnr": round(fnr, 4), "fpr": round(fpr, 4)})

ablation_df = pd.DataFrame(results).sort_values("f1", ascending=False)
print("Top 10 configs (toxic label):")
print(ablation_df.head(10).to_string(index=False))

wandb.log({"ablation_table": wandb.Table(dataframe=ablation_df.head(20))})

In [ ]:
# ── Cell 8: Error analysis ────────────────────────────────────────────────
thresh_toxic = thresholds.get("toxic", 0.5)
ml_toxic_preds = (ml_probs[:, TOXIC_IDX] >= thresh_toxic).astype(int)
true_toxic     = true_labels[:, TOXIC_IDX]

# False negatives: toxic bị bỏ sót
fn_mask = (true_toxic == 1) & (ml_toxic_preds == 0)
fn_df   = test_df[fn_mask][["text"]].copy()
fn_df["ml_score"] = ml_probs[fn_mask, TOXIC_IDX]
fn_df.to_csv("false_negatives.csv", index=False)
print(f"False negatives (missed toxic): {fn_mask.sum()} → false_negatives.csv")

# False positives: clean bị flag nhầm
fp_mask = (true_toxic == 0) & (ml_toxic_preds == 1)
fp_df   = test_df[fp_mask][["text"]].copy()
fp_df["ml_score"] = ml_probs[fp_mask, TOXIC_IDX]
fp_df.to_csv("false_positives.csv", index=False)
print(f"False positives (over-flagged): {fp_mask.sum()} → false_positives.csv")

# In vài ví dụ false negative để debug
print("\nSample false negatives (toxic bị bỏ sót):")
print(fn_df.head(5).to_string(index=False))

wandb.finish()
print("\nEvaluation complete!")